# print_feature — side-by-side feature comparison of 2–4 Data Matrix prints

Pick **2 to 4 images**. The notebook shows them side by side, runs the existing
`print_feature.py` extractor on each one (the 119-row print-characteristic vector),
and shows **one table: one row per feature, one column per image**. All the features
come from `print_feature.py`; the notebook imports it unchanged and computes no features itself.

**How to use**

1. Edit `IMAGE_PATHS` in the config cell below (2–4 entries), then *Run All*.
2. An entry can be
   * a crop file name from `dm_crops/` (e.g. `'IMG_0034_01.png'`),
   * any path to a crop or a full photo (absolute, or relative to this folder, e.g. `'images/IMG_0066.HEIC'`),
   * a tuple `(path, roi_index)` to choose which detected code of a full photo to use (default `ROI_INDEX`).

**Accepted inputs**

* **Crops** (files in `dm_crops/`, or any PNG with the crop text metadata). These are made by
  `dm_detector.crop(img, roi, annotate=False, pad=0.2)`: padded 20 %, deskewed and upright.
  Pixels per module and polarity come from the PNG text metadata, or else from `crop_labels.json` / `.csv`
  in the crop's folder or `CROPS_DIR`. If a crop has no labels, `dm_detector.detect` runs on the crop
  (as is, then padded white, then black) and uses the top ROI's `module_px`. As a last resort,
  `dm_detector.module_size` runs on the centred code square.
  The crop goes to `pf.extract_print_features` directly.
* **Full photos** (HEIC / JPG / PNG / …): `d.load_photo` → `d.detect` → ROI number `ROI_INDEX`
  (0 = top-ranked) → `pf.features_for_roi`.

**Empty (NaN) values mean "undefined", not zero**, for example:

* `dot_*`: these only fire on dotted prints (dot-peen / dotted laser / some inkjet). On solid prints they are NaN.
* `edge_bias_lead_trail_diff`: always NaN, because `print_feature` drops it (`pf.DROPPED_FEATURES`).
* `adain_*` (12 rows): NaN unless `INCLUDE_ADAIN = True`.
* `jpeg_blockiness`: needs the native, un-deskewed photo pixels. It is NaN for **crops** and only filled for full photos.
* `module_pitch_anisotropy`: needs the detector's x/y pitch. It is NaN for labelled crops unless `REDETECT_CROPS = True`.
* Other rows can be NaN when a code has too few usable pixels for a measurement (e.g. no voids, no straight edge runs).

**Nothing is written to disk.** The notebook only reads images and label files. Byte-code caching is
switched off, so importing the modules does not even create `__pycache__`. The only exception is
`INCLUDE_ADAIN = True` on a machine without the VGG-19 weights: torch then downloads them once to its cache.


## 1. Configuration  ← edit here

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()                 # the notebook lives in DM_detector/, next to print_feature.py
CROPS_DIR = PROJECT_DIR / 'dm_crops'     # crops from dm_detector.crop(..., annotate=False, pad=0.2)

# 2-4 entries: crop names in CROPS_DIR, paths (crop or full photo), or (path, roi_index) tuples
IMAGE_PATHS = [
    'IMG_0034_01.png',     # inkjet (DOD), leaflet
    'IMG_0063_01.png',     # thermal transfer, SAP strip
    'IMG_0050_01.png',     # laser ablation, light-on-dark
    # 'images/IMG_0066.HEIC',          # a full photo: detect -> top ROI -> features_for_roi
    # ('images/IMG_0069.HEIC', 1),     # a full photo, second detected code
]

ROI_INDEX = 0            # full photos: which detected code to use (0 = top-ranked)
INCLUDE_ADAIN = False    # True adds the 12 adain_* VGG-19 style features (needs torch + torchvision, slower)
REDETECT_CROPS = False   # True also runs d.detect on labelled crops (fills module_pitch_anisotropy + dotted hint)
SHOW_OVERLAY = True      # second figure row: print_feature's ink/substrate/edge-class mask overlay
SHOW_NAN_ROWS = True     # False hides features that are NaN for every image
HIGHLIGHT = 'relative'   # 'relative' (colour = deviation from row mean / row max|v|), 'row' (min-max per row), None
PRECISION = 3            # decimals in the tables
TOP_N = 15               # rows in the 'top differing features' table


## 2. Setup and input validation

In [ ]:
import sys, json, math, time, warnings
sys.dont_write_bytecode = True           # do not create __pycache__ next to the .py files
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image

import dm_detector as d
import print_feature as pf

FEATURE_NAMES = pf.feature_names()
CODE_PAD = pf.PrintFeatureConfig().code_pad          # 0.2, same padding as the dm_crops PNGs
print('print_feature:', Path(pf.__file__).resolve())
print('dm_detector  :', Path(d.__file__).resolve())
print(f'{len(FEATURE_NAMES)} features, code_pad={CODE_PAD}, dropped={pf.DROPPED_FEATURES}')
if INCLUDE_ADAIN:
    try:
        import torch, torchvision  # noqa: F401
    except ImportError as exc:
        raise ImportError('INCLUDE_ADAIN = True needs torch + torchvision; install them or set INCLUDE_ADAIN = False') from exc


def resolve_entry(entry):
    # IMAGE_PATHS entry -> (resolved Path, roi_index or None)
    roi_idx = None
    if isinstance(entry, (tuple, list)):
        if len(entry) != 2:
            raise ValueError(f'tuple entries must be (path, roi_index), got {entry!r}')
        entry, roi_idx = entry
        roi_idx = int(roi_idx)
    p = Path(entry).expanduser()
    candidates = [p] if p.is_absolute() else [CROPS_DIR / p, PROJECT_DIR / p, Path.cwd() / p]
    for c in candidates:
        if c.is_file():
            return c.resolve(), roi_idx
    raise FileNotFoundError(f'image not found: {entry!r} (looked in: ' + ', '.join(str(c) for c in candidates) + ')')


if not isinstance(IMAGE_PATHS, (list, tuple)) or not 2 <= len(IMAGE_PATHS) <= 4:
    n = len(IMAGE_PATHS) if isinstance(IMAGE_PATHS, (list, tuple)) else 'not a list'
    raise ValueError(f'IMAGE_PATHS must contain 2 to 4 images, got {n}. Edit the configuration cell.')
INPUTS = [resolve_entry(e) for e in IMAGE_PATHS]
for (p, r), e in zip(INPUTS, IMAGE_PATHS):
    print(f'  {p}' + (f'  (roi {r})' if r is not None else ''))


## 3. Input handling and feature extraction

For each image, this step decides whether it is a **crop** or a **full photo**:

* a crop is a file that lives in `CROPS_DIR`, carries the crop PNG metadata (`source_image`), or
  is listed in `crop_labels.json`/`.csv`;
* anything else is treated as a photo: the detector runs on it and the chosen ROI is cropped by `print_feature` itself.

The metadata kept for each image (source, method, px/module and where it came from, polarity, label) is shown
in the header table in section 5.


In [ ]:
_LABEL_CACHE = {}

def _label_table(folder):
    # crop_labels.json (preferred) or crop_labels.csv of one folder -> {crop_file: {...}}; {} if absent
    folder = Path(folder)
    if folder in _LABEL_CACHE:
        return _LABEL_CACHE[folder]
    table = {}
    js, cs = folder / 'crop_labels.json', folder / 'crop_labels.csv'
    try:
        if js.is_file():
            raw = json.loads(js.read_text())
            if isinstance(raw, dict):
                table = {str(k): dict(v) for k, v in raw.items()}
            elif isinstance(raw, list):
                table = {str(r.get('crop_file')): dict(r) for r in raw if r.get('crop_file')}
        elif cs.is_file():
            table = {str(r['crop_file']): r for r in pd.read_csv(cs).to_dict('records')}
    except Exception as exc:          # a half-written label file must not break the notebook
        warnings.warn(f'could not read labels in {folder}: {exc}')
        table = {}
    _LABEL_CACHE[folder] = table
    return table


def _clean(v):
    if v is None:
        return None
    if isinstance(v, float) and math.isnan(v):
        return None
    if isinstance(v, str) and v.strip().lower() in ('', 'nan', 'none', 'null'):
        return None
    return v


def _to_float(v):
    v = _clean(v)
    try:
        v = float(v)
    except (TypeError, ValueError):
        return None
    return v if math.isfinite(v) and v > 0 else None


def crop_metadata(path):
    # Labels of a crop: PNG text metadata first, then crop_labels.json/.csv; also tells where each came from
    png = {}
    try:
        with Image.open(path) as im:
            png = {k: v for k, v in (getattr(im, 'text', None) or {}).items()}
    except Exception:
        pass
    lab = {}
    for folder in dict.fromkeys([path.parent, CROPS_DIR]):
        lab = _label_table(folder).get(path.name, {})
        if lab:
            break
    meta, src = {}, {}
    keys = ['source_image', 'roi_index', 'bbox', 'px_per_module', 'polarity', 'decoded_text',
            'print_type', 'confidence', 'evidence', 'item_group']
    alias = {'bbox': ('bbox', 'bbox_xywh'), 'px_per_module': ('px_per_module', 'module_px')}
    for k in keys:
        for name, store in (('png metadata', png), ('crop_labels', lab)):
            for a in alias.get(k, (k,)):
                v = _clean(store.get(a))
                if v is not None:
                    meta[k], src[k] = v, name
                    break
            if k in meta:
                break
    return meta, src, bool(png.get('source_image')) or bool(lab)


def is_crop(path, has_labels):
    try:
        in_crops = path.parent.resolve() == CROPS_DIR.resolve()
    except OSError:
        in_crops = False
    return in_crops or has_labels


def detect_on_crop(img):
    # d.detect on the crop as is, then padded with white, then with black -> (top ROI, how) or (None, None)
    pad = max(16, int(0.25 * max(img.shape[:2])))
    trials = [('crop', img)]
    for name, val in (('white-padded crop', 255), ('black-padded crop', 0)):
        trials.append((name, cv2.copyMakeBorder(img, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(val,) * 3)))
    for name, im in trials:
        rois = d.detect(im, max_results=3)
        if rois:
            return rois[0], name
    return None, None


def module_size_on_centre(img):
    # Last resort: dm_detector.module_size on the centred code square (crop geometry pad=CODE_PAD)
    H, W = img.shape[:2]
    side = min(H, W) / (1.0 + 2.0 * CODE_PAD)
    cx, cy, h = W / 2.0, H / 2.0, side / 2.0
    pol = pf.prepare_roi(img, side / 20.0).polarity          # print_feature's own polarity rule
    poly = np.array([[cx - h, cy - h], [cx + h, cy - h], [cx + h, cy + h], [cx - h, cy + h]], np.float32)
    roi = d.DMRoi(polygon=poly, bbox=(int(cx - h), int(cy - h), int(side), int(side)), rotation=0,
                  tilt_deg=0.0, score=0.0, polarity=pol, side_px=side)
    m = d.module_size(img, roi)
    return _to_float(m.get('module_px')), m


def analyse_crop(path, meta, src):
    img = d.load_photo(path)
    info = {'kind': 'crop', 'image_px': f'{img.shape[1]}x{img.shape[0]}'}
    ppm = _to_float(meta.get('px_per_module'))
    pol = meta.get('polarity') if meta.get('polarity') in ('dark_on_light', 'light_on_dark') else None
    info['px_per_module_source'] = src.get('px_per_module') if ppm else None
    info['polarity_source'] = src.get('polarity') if pol else None
    xy, dotted = None, None
    if ppm is None or REDETECT_CROPS:
        roi, how = detect_on_crop(img)
        if roi is not None:
            _, xy = pf.roi_geometry(roi)
            dotted = bool(roi.dotted_fallback)
            info['redetect'] = f'd.detect on {how}: score {roi.score:.2f}, {roi.module_px or float("nan"):.2f} px/mod'
            if ppm is None:
                ppm, info['px_per_module_source'] = pf.roi_module_px(roi), f'd.detect on {how}'
            if pol is None:
                pol, info['polarity_source'] = roi.polarity, f'd.detect on {how}'
        elif ppm is None:
            info['redetect'] = 'd.detect found no code (crop, white pad, black pad)'
    if ppm is None:
        ppm, m = module_size_on_centre(img)
        if ppm is not None:
            info['px_per_module_source'] = f"d.module_size on centre square ({m.get('module_method')}, conf {m.get('module_conf', 0):.2f})"
    if ppm is None:
        ppm = min(img.shape[:2]) / (1.0 + 2.0 * CODE_PAD) / 20.0
        info['px_per_module_source'] = 'GUESS: code side / 20 (no detection)'
    if pol is None:
        info['polarity_source'] = 'print_feature auto'
    feats, dbg = pf.extract_print_features(
        img, module_px=ppm, angle_deg=0.0, polarity=pol, module_px_xy=xy, dotted_hint=dotted,
        include_adain=INCLUDE_ADAIN, return_debug=True)
    info.update(method='crop -> pf.extract_print_features', px_per_module=ppm,
                polarity=dbg['prepared'].polarity, dotted_hint=dotted)
    return feats, dbg, img, info


def analyse_photo(path, roi_idx):
    img = d.load_photo(path)
    idx = ROI_INDEX if roi_idx is None else roi_idx
    rois = d.detect(img, max_results=max(5, idx + 1))
    if not rois:
        raise RuntimeError(f'{path.name}: dm_detector found no Data Matrix code. For a crop, put it in CROPS_DIR '
                           f'({CROPS_DIR}) or give it the crop PNG metadata / a crop_labels.json entry.')
    if idx >= len(rois):
        raise IndexError(f'{path.name}: ROI index {idx} requested but only {len(rois)} code(s) detected')
    roi = rois[idx]
    feats, dbg = pf.features_for_roi(img, roi, include_adain=INCLUDE_ADAIN, return_debug=True)
    shown = d.crop(img, roi, annotate=False, pad=CODE_PAD)
    info = {'kind': 'photo', 'image_px': f'{img.shape[1]}x{img.shape[0]}',
            'method': f'photo -> d.detect ROI {idx}/{len(rois)} -> pf.features_for_roi',
            'roi_index': idx, 'detector_score': round(float(roi.score), 3),
            'px_per_module': pf.roi_module_px(roi), 'px_per_module_source': 'd.detect',
            'polarity': roi.polarity, 'polarity_source': 'd.detect',
            'dotted_hint': bool(roi.dotted_fallback), 'rotation': roi.rotation}
    return feats, dbg, shown, info


def analyse(path, roi_idx):
    t0 = time.perf_counter()
    meta, src, has_labels = ({}, {}, False)
    if path.suffix.lower() == '.png':
        meta, src, has_labels = crop_metadata(path)
    if is_crop(path, has_labels) and roi_idx is None:
        feats, dbg, shown, info = analyse_crop(path, meta, src)
    else:
        feats, dbg, shown, info = analyse_photo(path, roi_idx)
    names = [k for k, _ in feats]
    assert names == FEATURE_NAMES, 'feature order differs from pf.feature_names()'
    fd = pf.features_to_dict(feats)
    info.update({'file': path.name, 'path': str(path), 'runtime_s': round(time.perf_counter() - t0, 2),
                 'n_nan': int(sum(1 for v in fd.values() if not math.isfinite(v)))})
    for k in ('source_image', 'print_type', 'confidence', 'decoded_text', 'item_group'):
        if meta.get(k) is not None:
            info[k] = meta[k]
    if info['kind'] == 'crop' and meta.get('roi_index') is not None:
        info['roi_index'] = meta['roi_index']
    return {'features': fd, 'debug': dbg, 'shown': shown, 'info': info}


RESULTS = []
for p, r in INPUTS:
    res = analyse(p, r)
    RESULTS.append(res)
    i = res['info']
    print(f"{i['file']:<22} {i['kind']:<5} {i['px_per_module']:6.2f} px/mod ({i['px_per_module_source']}), "
          f"{i['polarity']}, {i['n_nan']} NaN, {i['runtime_s']:.1f} s")

# unique column names (same file name twice -> suffix)
COLS, _seen = [], {}
for res in RESULTS:
    n = res['info']['file']
    _seen[n] = _seen.get(n, 0) + 1
    COLS.append(n if _seen[n] == 1 else f'{n} ({_seen[n]})')


## 4. Images side by side

Top row: the image that was analysed. For a crop this is the crop itself; for a photo it is the padded, deskewed ROI crop.
Bottom row (`SHOW_OVERLAY`): `print_feature`'s canonical-pitch (24 px/module) crop with its masks.
Ink interior is red, substrate is blue, and the edge band is coloured by edge class: leading green, trailing red,
orth_left cyan, orth_right magenta.


In [ ]:
def _title(res, col):
    i = res['info']
    t = col
    if i.get('print_type'):
        t += f"\n{i['print_type']}" + (f" ({i['confidence']})" if i.get('confidence') else '')
    elif i['kind'] == 'photo':
        t += f"\nphoto, ROI {i['roi_index']}"
    t += f"\n{i['px_per_module']:.1f} px/mod, {i['polarity'].replace('_', ' ')}"
    return t


def _overlay(dbg):
    P = dbg['prepared']
    thumb = getattr(pf, '_thumb', None)          # print_feature's own overlay helper (used by its HTML report)
    if thumb is not None:
        try:
            return thumb(P, dbg.get('classes') or {}, size=600)[1]
        except Exception:
            pass
    return pf.code_thumbnail(P)


n = len(RESULTS)
rows = 2 if SHOW_OVERLAY else 1
fig, axes = plt.subplots(rows, n, figsize=(4.2 * n, 4.6 * rows), squeeze=False)
for j, (res, col) in enumerate(zip(RESULTS, COLS)):
    axes[0, j].imshow(cv2.cvtColor(res['shown'], cv2.COLOR_BGR2RGB))
    axes[0, j].set_title(_title(res, col), fontsize=10)
    if SHOW_OVERLAY:
        axes[1, j].imshow(cv2.cvtColor(_overlay(res['debug']), cv2.COLOR_BGR2RGB))
        cls = res['debug'].get('classes') or {}
        ax_name = {0: 'x', 1: 'y'}.get(int(res['features']['print_axis']) if np.isfinite(res['features']['print_axis']) else -1, '?')
        axes[1, j].set_title(f"masks @24 px/mod | print axis {ax_name}, leading edge faces {cls.get('leading', '?')}", fontsize=9)
for ax in axes.ravel():
    ax.axis('off')
plt.tight_layout()
plt.show()


## 5. Metadata header

`px_per_module_source` / `polarity_source` show where the values came from: PNG metadata, crop_labels,
detection on the crop, or the photo's `d.detect` ROI. `n_nan` counts the undefined features of each image.


In [ ]:
META_ROWS = ['kind', 'method', 'source_image', 'roi_index', 'image_px', 'px_per_module', 'px_per_module_source',
             'polarity', 'polarity_source', 'dotted_hint', 'redetect', 'detector_score', 'print_type', 'confidence',
             'item_group', 'decoded_text', 'n_nan', 'runtime_s']
meta_df = pd.DataFrame({col: {k: res['info'].get(k) for k in META_ROWS} for res, col in zip(RESULTS, COLS)})
meta_df = meta_df.loc[[k for k in META_ROWS if meta_df.loc[k].notna().any()]]
meta_df.loc['px_per_module'] = [f'{v:.2f}' if isinstance(v, (int, float)) else v for v in meta_df.loc['px_per_module']]
meta_df = meta_df.astype(object).where(meta_df.notna(), '')
meta_df.index.name = 'metadata'
display(meta_df)


## 6. Feature table (one row per feature, one column per image)

The rows follow `pf.feature_names()` order. `group` comes from the name prefix.
`range` = max − min across the images. `rel_spread` = range / max|value| (0 = identical,
1 = 100 % of the largest magnitude, 2 = equal magnitude with opposite signs). With `HIGHLIGHT = 'relative'`,
each cell is coloured by (value − row mean) / max|value| on a fixed −1…+1 scale (blue = below the row mean, red = above),
so rows that barely differ stay white. Tiny non-zero values (e.g. Gabor energies) are shown in scientific notation.
With `'row'`, each row is stretched from its own min to max, which is also the case for near-identical values.
Note that angle features (`*_orientation*`, degrees relative to the print axis) and signed features (skews, log2 ratios)
can reach a large `rel_spread` near zero.


In [ ]:
_GROUPS = [
    ('capture_qc', ('capture_px_per_module', 'resample_factor', 'upsampled_flag', 'module_pitch_anisotropy',
                    'sharpness_score', 'glare_fraction', 'jpeg_blockiness', 'bimodality_separability')),
    ('print_axis', ('mark_polarity', 'print_axis', 'print_polarity')),
    ('edge', ('edge_', 'corner_')), ('structure_tensor', ('st_',)), ('gabor', ('gabor_',)),
    ('banding', ('banding_',)), ('lbp', ('lbp_', 'clbp_', 'ldp_', 'hog_')), ('glcm', ('glcm_',)),
    ('ink', ('ink_',)), ('substrate', ('substrate_',)), ('spatter', ('spatter_',)), ('dot', ('dot_',)),
    ('adain', ('adain_',)),
]

def feature_group(name):
    for g, prefixes in _GROUPS:
        if name.startswith(prefixes):
            return g
    return 'other'


vals = pd.DataFrame({col: [res['features'][k] for k in FEATURE_NAMES] for res, col in zip(RESULTS, COLS)},
                    index=pd.Index(FEATURE_NAMES, name='feature'), dtype=float)
n_valid = vals.notna().sum(axis=1)
rng = vals.max(axis=1) - vals.min(axis=1)
mag = vals.abs().max(axis=1)
rel = (rng / mag).where(mag > 1e-12, 0.0).where(n_valid >= 2)

feat_df = vals.copy()
feat_df.insert(0, 'group', [feature_group(k) for k in FEATURE_NAMES])
feat_df['range'] = rng.where(n_valid >= 2)
feat_df['rel_spread'] = rel
shown_df = feat_df if SHOW_NAN_ROWS else feat_df[n_valid > 0]
print(f'{len(shown_df)} of {len(feat_df)} features shown; all-NaN rows: {int((n_valid == 0).sum())} '
      f'({"shown" if SHOW_NAN_ROWS else "hidden"}); rows with a NaN in some image: {int(((n_valid > 0) & (n_valid < len(COLS))).sum())}')


def _css(rgba):
    r, g, b = [int(255 * c) for c in rgba[:3]]
    lum = 0.299 * r + 0.587 * g + 0.114 * b
    return f'background-color: rgb({r},{g},{b}); color: {"black" if lum > 140 else "white"}'


def _fmt(v):
    # fixed decimals; tiny non-zero magnitudes (e.g. Gabor energies) in scientific notation
    if v is None or not np.isfinite(v):
        return 'NaN'
    return f'{v:.{PRECISION - 1}e}' if 0 < abs(v) < 10 ** -PRECISION else f'{v:.{PRECISION}f}'


def _relative_colors(row, cmap=plt.get_cmap('RdBu_r'), scale=1.0):
    v = row.astype(float)
    m = np.nanmax(np.abs(v.values)) if v.notna().sum() >= 2 else np.nan
    if not np.isfinite(m) or m <= 1e-12:
        return ['' for _ in v]
    t = ((v - np.nanmean(v.values)) / m / scale).clip(-1, 1)
    return ['' if not np.isfinite(x) else _css(cmap(0.5 + 0.5 * x)) for x in t]


def style_table(df, cols):
    st = df.style.format(_fmt, subset=[c for c in df.columns if c != 'group'])
    if HIGHLIGHT == 'relative':
        st = st.apply(_relative_colors, axis=1, subset=cols)
    elif HIGHLIGHT == 'row':
        st = st.background_gradient(cmap='RdBu_r', axis=1, subset=cols)
    if 'rel_spread' in df.columns:
        st = st.bar(subset=['rel_spread'], vmin=0, vmax=2, color='#f4a582')
    st = st.map(lambda v: 'color: #aaa', subset=pd.IndexSlice[df.index[df[cols].isna().all(axis=1)], cols])
    return st.set_properties(subset=['group'], **{'color': '#555', 'font-size': '90%'}) \
             .set_table_styles([{'selector': 'th.col_heading', 'props': 'text-align: center; white-space: pre;'}])


display(style_table(shown_df, COLS))


## 7. Top differing features

This is `rel_spread` sorted in descending order, over features that are defined for at least 2 images. The `capture_qc` group
(pixels per module, resampling, glare, sharpness …) describes the capture and not the print, so it is left out here.
It is still in the full table above.


In [ ]:
top = feat_df[(feat_df['group'] != 'capture_qc') & feat_df['rel_spread'].notna()]
top = top.sort_values('rel_spread', ascending=False).head(TOP_N)
display(style_table(top, COLS))


## Notes and limitations

* **Crops and jpeg_blockiness:** a crop is deskewed and resampled, so the native 8×8 lattice is lost and `jpeg_blockiness`
  is NaN for crops. The iPhone photos are HEIC anyway, which makes it only a weak indicator (see the `print_feature.py` docstring).
* **Crops and the canonical geometry:** the code square is assumed to be centred with `code_pad = 0.2`, as in `dm_crops`.
  A crop with other padding shifts the analysis region.
* **Polarity:** when no label or detection gives the polarity, `print_feature`'s automatic rule decides.
  It can be wrong on crops with a large dark or light surround. Check the `polarity` row in the header table.
* **Lead/trail features:** when `print_polarity_confidence` is ~0, the lead/trail sign is effectively arbitrary.
  Treat those asymmetry features as unordered (see the `print_feature.py` docstring).
* **Comparing codes captured at very different px/module:** everything is resampled to 24 px/module, but
  upsampled codes (`upsampled_flag = 1`) have less real detail. Prefer comparing codes with similar `capture_px_per_module`.
